# 使用 Weaviate 与 Generative OpenAI 模块进行生成式搜索

本 notebook 适用于以下场景：
* 你的数据已经在 Weaviate 中
* 你希望将 Weaviate 与 Generative OpenAI 模块（[generative-openai](https://weaviate.io/developers/weaviate/modules/reader-generator-modules/generative-openai)）结合使用。



## 前置条件

本 cookbook 只涵盖生成式搜索示例，不涉及配置和数据导入。

为了充分利用本 cookbook，请先完成 [入门 cookbook](./getting-started-with-weaviate-and-openai.ipynb)，你将在其中学习使用 Weaviate 的基础知识并导入演示数据。

检查清单：
* 已完成 [入门 cookbook](./getting-started-with-weaviate-and-openai.ipynb)，
* 已创建一个 `Weaviate` 实例，
* 已将数据导入你的 `Weaviate` 实例，
* 你有一个 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)

===========================================================
## 准备你的 OpenAI API 密钥

`OpenAI API key` 用于在导入时对数据进行向量化，以及运行查询。

如果你还没有 OpenAI API 密钥，可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取。

拿到密钥后，请将其添加到环境变量 `OPENAI_API_KEY` 中。

In [ ]:
# Export OpenAI API Key
!export OPENAI_API_KEY="your key"

In [ ]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = 'your-key-goes-here'

if os.getenv("OPENAI_API_KEY") is not None:
    print ("OPENAI_API_KEY is ready")
else:
    print ("OPENAI_API_KEY environment variable not found")

## 连接到你的 Weaviate 实例

在本节中，我们将：

1. 检查环境变量 `OPENAI_API_KEY` ——**请确保**你已完成 [#准备你的 OpenAI API 密钥](#准备你的-openai-api-密钥) 中的步骤
2. 使用你的 `OpenAI API Key` 连接到 Weaviate
3. 并测试客户端连接

### 客户端 

完成这一步后，`client` 对象将用于执行所有与 Weaviate 相关的操作。

In [ ]:
import weaviate
from datasets import load_dataset
import os

# Connect to your Weaviate instance
client = weaviate.Client(
    url="https://your-wcs-instance-name.weaviate.network/",
    # url="http://localhost:8080/",
    auth_client_secret=weaviate.auth.AuthApiKey(api_key="<YOUR-WEAVIATE-API-KEY>"), # comment out this line if you are not using authentication for your Weaviate instance (i.e. for locally deployed instances)
    additional_headers={
        "X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")
    }
)

# Check if your instance is live and ready
# This should return `True`
client.is_ready()

## 生成式搜索
Weaviate 提供了 [Generative Search OpenAI](https://weaviate.io/developers/weaviate/modules/reader-generator-modules/generative-openai) 模块，它会基于你 Weaviate 实例中存储的数据生成响应。

构建生成式搜索查询的方式与 Weaviate 中标准的语义搜索查询非常相似。

例如：
* 在 "Articles" 中搜索，
* 返回 "title"、"content"、"url"
* 查找与 "football clubs" 相关的对象
* 将结果限制为 5 个对象

```
    result = (
        client.query
        .get("Articles", ["title", "content", "url"])
        .with_near_text("concepts": "football clubs")
        .with_limit(5)
        # generative query will go here
        .do()
    )
```

现在，你可以添加 `with_generate()` 函数来应用生成式转换。`with_generate` 接受以下任一参数：
- `single_prompt` ——为每个返回的对象分别生成一个响应，
- `grouped_task` ——基于所有返回的对象生成单个响应。


In [ ]:
def generative_search_per_item(query, collection_name):
    prompt = "Summarize in a short tweet the following content: {content}"

    result = (
        client.query
        .get(collection_name, ["title", "content", "url"])
        .with_near_text({ "concepts": [query], "distance": 0.7 })
        .with_limit(5)
        .with_generate(single_prompt=prompt)
        .do()
    )
    
    # Check for errors
    if ("errors" in result):
        print ("\033[91mYou probably have run out of OpenAI API calls for the current minute – the limit is set at 60 per minute.")
        raise Exception(result["errors"][0]['message'])
    
    return result["data"]["Get"][collection_name]

In [ ]:
query_result = generative_search_per_item("football clubs", "Article")

for i, article in enumerate(query_result):
    print(f"{i+1}. { article['title']}")
    print(article['_additional']['generate']['singleResult']) # print generated response
    print("-----------------------")

In [79]:
def generative_search_group(query, collection_name):
    generateTask = "Explain what these have in common"

    result = (
        client.query
        .get(collection_name, ["title", "content", "url"])
        .with_near_text({ "concepts": [query], "distance": 0.7 })
        .with_generate(grouped_task=generateTask)
        .with_limit(5)
        .do()
    )
    
    # Check for errors
    if ("errors" in result):
        print ("\033[91mYou probably have run out of OpenAI API calls for the current minute – the limit is set at 60 per minute.")
        raise Exception(result["errors"][0]['message'])
    
    return result["data"]["Get"][collection_name]

In [ ]:
query_result = generative_search_group("football clubs", "Article")

print (query_result[0]['_additional']['generate']['groupedResult'])

感谢你的跟随学习，现在你已经具备了搭建自己的向量数据库并利用嵌入实现各种酷炫功能的能力——尽情享受吧！对于更复杂的用例，请继续学习本仓库中的其他 cookbook 示例。